# Lab: Advanced AI Agent Concepts

This lab teaches five core ideas behind learning agents by having you **build the
key logic yourself**, not just run finished code. For each concept you get a working
harness (the setup, the examples, the visualization). The important part, the actual
learning rule, is left blank for you to write.

### How to use this notebook
- Cells marked `# TODO` are yours to complete. They will raise
  `NotImplementedError` until you finish them.
- Run cells top to bottom with **Shift + Enter**. Later parts reuse earlier code,
  so order matters.
- After each part there is an **Experiment** cell (change something and rerun) and a
  **Reflection** cell (answer in your own words, based on what you saw).
- Do not use a text AI to write your code or your answers. Your explanations must
  match the output your own code produced.


In [1]:
# Setup (ready to run)
import random
print("Ready.")

Ready.


## Part 1 — Deep Q-Network (DQN)

A DQN learns the **value** of taking an action in a state. Here we store those values
in a Q-table (a dictionary mapping `(state, action)` to a number). After each
experience, we nudge the stored value toward what we actually observed.

**The Q-learning update formula:**
```
new_value = old_value + learning_rate * (reward + discount_factor * next_max - old_value)
```
where `next_max` is the best Q-value available from the next state.

**Your task (Cell below):** implement the `update` method using that formula.


In [13]:
class SimpleDQN:
    def __init__(self):
        self.q_table = {}            # (state, action) -> value
        self.learning_rate = 0.1
        self.discount_factor = 0.9

    def get_q_value(self, state, action):
        # Ready to use. Returns 0.0 for unseen pairs.
        return self.q_table.get((state, action), 0.0)

    def update(self, state, action, reward, next_state):
        # TODO: implement the Q-learning update.
        old_value = self.get_q_value(state, action)
        next_max  = max(self.get_q_value(next_state, a) for a in ['move', 'stop'])
        new_value = old_value + self.learning_rate * (reward + self.discount_factor * next_max - old_value)
        self.q_table[(state, action)] = new_value

In [19]:
# Run this to test your update (provided)
agent = SimpleDQN()
agent.update("obstacle_ahead", "stop", reward=10, next_state="clear_path")
print("After 1 update:", agent.get_q_value("obstacle_ahead", "stop"))

# Apply the SAME experience four more times and watch the value climb
for _ in range(4):
    agent.update("obstacle_ahead", "stop", reward=10, next_state="clear_path")
    print("Q-value:", round(agent.get_q_value("obstacle_ahead", "stop"), 4))

After 1 update: 1.0
Q-value: 1.9
Q-value: 2.71
Q-value: 3.439
Q-value: 4.0951


In [18]:
# EXPERIMENT (TODO)
# Make a new agent, set its learning_rate to 0.5, and repeat the same five updates.
# Compare how fast the value rises against the 0.1 agent above.

# TODO: write your experiment here.
fast_agent = SimpleDQN()
fast_agent.learning_rate = 0.5

print("learning_rate = 0.5")
for i in range(5):
    fast_agent.update("obstacle_ahead", "stop", reward=10, next_state="clear_path")
    print(f"Update {i+1}: Q-value = {round(fast_agent.get_q_value('obstacle_ahead', 'stop'), 4)}")


learning_rate = 0.5
Update 1: Q-value = 5.0
Update 2: Q-value = 7.5
Update 3: Q-value = 8.75
Update 4: Q-value = 9.375
Update 5: Q-value = 9.6875


**Reflection 1.** In two to three sentences, explain why the Q-value rose with each
repeat, and what changing the learning rate did to how fast it rose.

_Type your answer here._

Each repeat pushed the Q-value toward the target by using reward + discount x * next_max = 10. With each update the gap shrank and the value climbed higher closer to 10 without overshooting.

Increasing the learning rate changed how fast the value climbed.

At 0.1 it slowly went from 1.0 -> 1.9 -> 2.71 -> 3.43 -> 4.09 while a 0.5 accelerated the learning rate to 5 -> 7.5 -> 8.75 -> 9.37 -> 9.68.

A higher learning rate reaches the target in fewer repeats.

## Part 2 — Policy Gradient

A policy-gradient agent does not store values. It stores a **probability** for each
action and shifts those probabilities toward actions that earned reward.

**Your task:** implement `update_policy` so that when an action earns positive reward,
its probability goes up, and then all probabilities are renormalized to sum to 1.


In [22]:
class SimplePolicyAgent:
    def __init__(self):
        self.action_probabilities = {'move': 0.5, 'stop': 0.5}
        self.learning_rate = 0.1

    def choose_action(self):
        # Ready to use. Samples an action using the current probabilities.
        return random.choices(
            list(self.action_probabilities.keys()),
            list(self.action_probabilities.values())
        )[0]

    def update_policy(self, action, reward):
        if reward > 0:
            self.action_probabilities[action] += self.learning_rate * reward
            total = sum(self.action_probabilities.values())
            for a in self.action_probabilities:
                self.action_probabilities[a] /= total

In [23]:
# Run this to test (provided). Reward 'move', punish 'stop'.
policy_agent = SimplePolicyAgent()
for _ in range(8):
    a = policy_agent.choose_action()
    reward = 1 if a == 'move' else -1
    policy_agent.update_policy(a, reward)
    print({k: round(v, 3) for k, v in policy_agent.action_probabilities.items()})

{'move': 0.5, 'stop': 0.5}
{'move': 0.5, 'stop': 0.5}
{'move': 0.545, 'stop': 0.455}
{'move': 0.587, 'stop': 0.413}
{'move': 0.624, 'stop': 0.376}
{'move': 0.658, 'stop': 0.342}
{'move': 0.658, 'stop': 0.342}
{'move': 0.658, 'stop': 0.342}


In [24]:
# EXPERIMENT (TODO)
# Copy the loop above but FLIP the reward rule so 'stop' is rewarded instead.
# Run it and watch which way the probabilities drift.

# TODO: write your experiment here.
policy_agent = SimplePolicyAgent()
for _ in range(8):
    a = policy_agent.choose_action()
    reward = 1 if a == 'stop' else -1
    policy_agent.update_policy(a, reward)
    print({k: round(v, 3) for k, v in policy_agent.action_probabilities.items()})

{'move': 0.5, 'stop': 0.5}
{'move': 0.455, 'stop': 0.545}
{'move': 0.413, 'stop': 0.587}
{'move': 0.376, 'stop': 0.624}
{'move': 0.342, 'stop': 0.658}
{'move': 0.31, 'stop': 0.69}
{'move': 0.282, 'stop': 0.718}
{'move': 0.257, 'stop': 0.743}


**Reflection 2.** In two to three sentences, describe how the action probabilities
changed once you flipped the reward, and how this differs from how the DQN in Part 1
learned.

_Type your answer here._

Flipping the reward causes the probabilities to drift opposite of eachother.

'stop' rose to 0.743 and 'move' fell to 0.257 when it mirrored the original run.

The DQN stores a value for each action and updates it based on the observed reward, while the policy agent stores only probabilities and updates them toward the valued action before renormalizing the sum to 1.

## Part 3 — Multi-Agent System

Several agents share one world. Each can **sense**, **decide**, and **act**. They also
**communicate** (here, just by seeing where the others are) so they can avoid collisions.

The agent, the world, and the visualization are provided. **Your task** is the
coordination rule: an agent should not move forward if another agent is in the cell
directly ahead of it.


In [53]:
# Provided: a single agent, the world, and a text visualizer.
class SimpleAgent:
    def __init__(self, world_size):
        self.position = random.randint(0, world_size - 1)
        self.world_size = world_size

    def sense(self, world):
        return world[self.position]

    def decide(self, observation):
        return 'move' if observation == ' ' else 'stop'

    def act(self, action):
        if action == 'move' and self.position < self.world_size - 1:
            self.position += 1

def create_world(size):
    return [' ' for _ in range(size)]

def visualize(world, positions):
    cells = []
    for i in range(len(world)):
        if i in positions:
            cells.append(str(positions.index(i) + 1))
        else:
            cells.append(world[i])
    return "[" + "][".join(cells) + "]"

print("Helpers ready.")

Helpers ready.


In [54]:
class SimpleMultiAgentSystem:
    def __init__(self, num_agents=3, world_size=10):
        self.agents = [SimpleAgent(world_size) for _ in range(num_agents)]
        self.world_size = world_size
        self.world = create_world(world_size)

    def communicate(self, agent_index):
        # Ready to use. Returns the positions of all the OTHER agents.
        return [a.position for i, a in enumerate(self.agents) if i != agent_index]

    def coordinate_actions(self):
        for i, agent in enumerate(self.agents):
            other_positions = self.communicate(i)
            observation = agent.sense(self.world)

            # Coordination rule: stop if another agent is directly ahead
            if agent.position + 1 in other_positions:
                action = 'stop'
            else:
                action = agent.decide(observation)
            agent.act(action)

In [58]:
# Run this to test (provided)
system = SimpleMultiAgentSystem(num_agents=3)
for step in range(5):
    system.coordinate_actions()
    positions = [a.position for a in system.agents]
    print(f"Step {step+1}: {visualize(system.world, positions)}")

Step 1: [ ][ ][ ][ ][ ][1][ ][3][2][ ]
Step 2: [ ][ ][ ][ ][ ][ ][1][ ][3][2]
Step 3: [ ][ ][ ][ ][ ][ ][ ][1][3][2]
Step 4: [ ][ ][ ][ ][ ][ ][ ][1][3][2]
Step 5: [ ][ ][ ][ ][ ][ ][ ][1][3][2]


In [64]:
# EXPERIMENT (TODO)
# Run the same simulation again with num_agents = 5 and watch how often agents stop.

# TODO: write your experiment here.
# EXPERIMENT — five agents instead of three
system = SimpleMultiAgentSystem(num_agents=5)
for step in range(5):
    system.coordinate_actions()
    positions = [a.position for a in system.agents]
    print(f"Step {step+1}: {visualize(system.world, positions)}")

Step 1: [ ][3][ ][4][ ][ ][ ][5][1][2]
Step 2: [ ][ ][3][ ][4][ ][ ][5][1][2]
Step 3: [ ][ ][ ][3][ ][4][ ][5][1][2]
Step 4: [ ][ ][ ][ ][3][ ][4][5][1][2]
Step 5: [ ][ ][ ][ ][ ][3][4][5][1][2]


**Reflection 3.** In two to three sentences, describe what the coordination rule does
when agents get close, and what changed when you used five agents instead of three.

_Type your answer here._

The coordination rule makes an agent stop instead of moving forward whenever another agent is in the cell ahead of it which prevents collisions.

Three agents moved like a convoy, but five agents in the same 10-cell space seemed to collide.

With more agents packed in, the cell ahead was occupied far more often, which led to agents stopping more frequently and the group colliding.

## Part 4 — Federated Learning

Several agents each learn on their own, then **share knowledge by averaging** their
Q-tables, without ever sharing raw experience. After aggregation, every agent holds the
same averaged values.

**Your task:** implement `aggregate_knowledge` to average the Q-values across all agents
and write the averaged table back into each one. (Reuses `SimpleDQN` from Part 1.)


In [65]:
class SimpleFederatedSystem:
    def __init__(self, num_agents=3):
        self.agents = [SimpleDQN() for _ in range(num_agents)]

    def aggregate_knowledge(self):
        # TODO:
        # 1. Collect every (state, action) key that appears in ANY agent's q_table.
        all_keys = set()
        for agent in self.agents:
            all_keys.update(agent.q_table.keys())
        # 2. For each key, average that value across all agents (use 0.0 if an agent
        #    has not seen it).
        averaged = {}
        for key in all_keys:
            total = sum(agent.q_table.get(key, 0.0) for agent in self.agents)
            averaged[key] = total / len(self.agents)
        # 3. Replace every agent's q_table with a copy of the averaged table.
        for agent in self.agents:
            agent.q_table = averaged.copy()

In [66]:
# Run this to test (provided)
fed = SimpleFederatedSystem(num_agents=3)
for i, agent in enumerate(fed.agents):
    agent.q_table[(f"position_{i}", "move")] = round(random.random(), 3)

print("Before aggregation:")
for i, agent in enumerate(fed.agents):
    print(f"  Agent {i}: {agent.q_table}")

fed.aggregate_knowledge()

print("\nAfter aggregation:")
for i, agent in enumerate(fed.agents):
    print(f"  Agent {i}: {agent.q_table}")

Before aggregation:
  Agent 0: {('position_0', 'move'): 0.204}
  Agent 1: {('position_1', 'move'): 0.497}
  Agent 2: {('position_2', 'move'): 0.865}

After aggregation:
  Agent 0: {('position_0', 'move'): 0.06799999999999999, ('position_1', 'move'): 0.16566666666666666, ('position_2', 'move'): 0.28833333333333333}
  Agent 1: {('position_0', 'move'): 0.06799999999999999, ('position_1', 'move'): 0.16566666666666666, ('position_2', 'move'): 0.28833333333333333}
  Agent 2: {('position_0', 'move'): 0.06799999999999999, ('position_1', 'move'): 0.16566666666666666, ('position_2', 'move'): 0.28833333333333333}


**Reflection 4.** In two to three sentences, explain what aggregation did to each
agent's values and give one real situation where sharing averaged knowledge, instead of
raw data, would be useful.

_Type your answer here._

Aggregation replaced each agent's Q-table with the same averaged table which made all three agents have identical values.

The key values became the mean for all agents.

This can be useful for hospitals training a shared diagnostic model. A hospital can average their model updates to benefit from combined learning without having to exchange private patient records.


## Part 5 — Compare Learning Approaches

Finally, run both kinds of agent over many episodes and compare their average reward.
The loop is provided, but **you must call the function** at the bottom, since the
original code defined it and never ran it.


In [69]:
def compare_learning_approaches(episodes=100):
    dqn_agent = SimpleDQN()
    policy_agent = SimplePolicyAgent()
    dqn_rewards, policy_rewards = [], []

    for episode in range(episodes):
        # DQN: pick the action with the highest current Q-value
        state, dqn_total = "start", 0
        for _ in range(5):
            action = max(['move', 'stop'], key=lambda a: dqn_agent.get_q_value(state, a))
            reward = random.choice([-1, 1])
            next_state = f"state_{random.randint(1, 5)}"
            dqn_agent.update(state, action, reward, next_state)
            dqn_total += reward
            state = next_state

        # Policy gradient
        policy_total = 0
        for _ in range(5):
            action = policy_agent.choose_action()
            reward = random.choice([-1, 1])
            policy_agent.update_policy(action, reward)
            policy_total += reward

        dqn_rewards.append(dqn_total)
        policy_rewards.append(policy_total)

        if episode % 10 == 0:
            print(f"Episode {episode:>3} | "
                  f"DQN avg {sum(dqn_rewards[-10:]) / 10:+.1f} | "
                  f"Policy avg {sum(policy_rewards[-10:]) / 10:+.1f}")

    return dqn_rewards, policy_rewards

In [68]:
# TODO: call compare_learning_approaches() and run it.
dqn_rewards, policy_rewards = compare_learning_approaches()

Episode   0 | DQN avg -0.1 | Policy avg +0.3
Episode  10 | DQN avg -0.2 | Policy avg +0.2
Episode  20 | DQN avg +0.2 | Policy avg -0.2
Episode  30 | DQN avg +0.0 | Policy avg +0.4
Episode  40 | DQN avg +1.0 | Policy avg -0.2
Episode  50 | DQN avg +1.0 | Policy avg +1.2
Episode  60 | DQN avg +0.6 | Policy avg +0.2
Episode  70 | DQN avg -0.8 | Policy avg -0.2
Episode  80 | DQN avg +0.4 | Policy avg +0.6
Episode  90 | DQN avg +0.0 | Policy avg +0.6


**Reflection 5.** In two to three sentences, state which approach showed steadier
average reward in your run and offer one reason why. (The reward here is random, so look
at the trend, not a single number.)

_Type your answer here._

The policy-gradient agent was slightly steadier ranging from -0.2 and +1.2.

The DQN averaged spread out wider ranging from -0.8 and 1.0.

The agent does not seem to be learning, the only difference is noise.

## Final Reflection and Submission

**Final reflection (150 to 250 words).** Which of the five concepts was clearest to you,
which was hardest, and what is one real situation where a multi-agent or federated
approach would beat a single agent? Reference specific output you saw.

_Type your answer here._

The DQN update in Part 1 was the clearest to understand. Watching the same experience push the Q-value up towards the target at different learning rates was easy to understand and see.

The hardest part was the comparison in Part 5. The reward was random, with no idea of the action taken. The DQN swung from -0.8 to 1.0 while the policy agent held between -0.2 and +0.6. The difficulty was recognizing that neither agent was really learning and was just due to luck, as it was affected by the generated noise.

A real situation where a federated approach beats a single agent would be in hospitals training a shared diagnostic model. In part 4, the agents each held a single value, and after aggregation, all three held the same averaged table. This would help in a hospital environment, as they can benefit from others' learning through average updates without exchanging private patient information.

---

### Before you submit
- [X] Every `# TODO` is implemented and no cell raises `NotImplementedError`.
- [X] All cells run top to bottom with outputs visible.
- [X] Both experiment cells (Parts 1, 2, 3, 5) contain your changes and their output.
- [X] All six reflections are answered in your own words.
- [X] Export to PDF with **File > Print > Save as PDF** and submit on Canvas.
